# ML-09 — Validation and Research Claim Audit

This notebook audits the Week-5 Logistic Regression model using the starter snapshot. The outcome is a **current-state decline proxy**, not a future outcome: `decline_proxy_rule = (trend_direction == 'down')`. Results are measured associations for decision-support practice, not causal or production-performance claims.

## 1. Two paper findings + my methodology questions

### Finding 1 — the paper reports that its Random Forest reached ROC AUC 0.767 and 100% Precision@50 on a client-grouped test.

**Constructive methodology question:** The paper defines decline as a ≥20% drop in impressions in the last 30 days versus the prior 30 days. I would ask the authors to show, for the exact held-out clients, the label base rate and the number of positive pages behind Precision@50, and to confirm that every scoring feature ends before the last-30-day label window. Those details would help readers judge whether the top-50 result reflects usable prioritization signal rather than an overlapping measurement window or a favorable prevalence. The client-grouped design is a meaningful check for generalization to unseen clients, but it does not by itself establish performance on a future period.

### Finding 2 — the paper reports that visibility alone contributed +0.17 ROC AUC and that it was the dominant signal.

**Constructive methodology question:** I would ask whether the visibility-only ablation, the full model, and the comparison models used identical held-out clients, preprocessing fit only on the training clients, and the same label threshold. I would also ask for uncertainty across several client-grouped folds rather than one split. That would show whether the measured lift is stable across clients and whether the reported hierarchy is sensitive to one held-out group. This question does not dispute the observed result; it asks for the validation evidence needed to interpret its scope.

## 2. My model under an honest split (before/after)

The **before** evaluation randomly splits rows, so pages from a client can appear in both training and test data. The **after** evaluation holds out whole pseudonymized clients with `GroupShuffleSplit`; `client_id` is used only to form groups and never enters the model. Both evaluations use the same model, features, seed, and current-state proxy. I report the test base rate beside each metric because precision and accuracy have no useful meaning without it.

A client-grouped split is more honest for the stated generalization question—whether a score trained on some clients transfers to unseen clients. It is still not a time-aware deployment simulation because the starter file is a snapshot.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists())
df = pd.read_csv(repo_root / 'data/raw/content_refresh_anonymized.csv')
target_col = 'decline_proxy_rule'
df[target_col] = df['trend_direction'].eq('down').astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'impressions_90d',
    'clicks_90d', 'sessions_90d', 'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
print(f'Rows: {len(df):,}; pseudonymized clients: {df.client_id.nunique()}; overall proxy-decline rate: {df[target_col].mean():.1%}')
print(f'pandas {pd.__version__}; scikit-learn {sklearn.__version__}; seed {RANDOM_SEED}')

Rows: 30,000; pseudonymized clients: 32; overall proxy-decline rate: 54.2%
pandas 2.0.3; scikit-learn 1.3.2; seed 42


In [2]:
def make_model(columns):
    preprocess = ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler', StandardScaler()),
        ]), columns),
    ])
    return Pipeline([
        ('preprocess', preprocess),
        ('logistic', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)),
    ])

def precision_at_k(y_true, scores, k=20):
    ranked = pd.DataFrame({'y': np.asarray(y_true), 'score': np.asarray(scores)}).sort_values('score', ascending=False)
    return ranked.head(k)['y'].mean()

def evaluate_split(name, train_index, test_index, columns=feature_cols):
    model = make_model(columns)
    model.fit(df.iloc[train_index][columns], df.iloc[train_index][target_col])
    probability = model.predict_proba(df.iloc[test_index][columns])[:, 1]
    y_test = df.iloc[test_index][target_col]
    return {
        'split': name, 'test_rows': len(test_index),
        'test_clients': df.iloc[test_index].client_id.nunique(),
        'test_base_rate': y_test.mean(), 'roc_auc': roc_auc_score(y_test, probability),
        'average_precision': average_precision_score(y_test, probability),
        'precision_at_20': precision_at_k(y_test, probability, 20),
    }, model, probability

all_index = np.arange(len(df))
random_train, random_test = train_test_split(all_index, test_size=0.25, stratify=df[target_col], random_state=RANDOM_SEED)
group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
group_train, group_test = next(group_splitter.split(df, df[target_col], groups=df['client_id']))

random_metrics, _, _ = evaluate_split('Before: random row split', random_train, random_test)
group_metrics, grouped_model, grouped_probability = evaluate_split('After: client-grouped split', group_train, group_test)
comparison = pd.DataFrame([random_metrics, group_metrics])
for column in ['test_base_rate', 'roc_auc', 'average_precision', 'precision_at_20']:
    comparison[column] = comparison[column].map(lambda value: f'{value:.3f}')
display(comparison)
print('Interpretation: the difference between these rows is a measured sensitivity to the split design, not proof that one estimate is universally correct. The grouped result is the appropriate estimate for unseen-client use in this snapshot.')

,split,test_rows,test_clients,test_base_rate,roc_auc,average_precision,precision_at_20
0,Before: random row split,7500,31,0.542,0.683,0.701,1.000
1,After: client-grouped split,7115,8,0.517,0.589,0.600,0.850


Interpretation: the difference between these rows is a measured sensitivity to the split design, not proof that one estimate is universally correct. The grouped result is the appropriate estimate for unseen-client use in this snapshot.


## 3. Leakage audit

The target is derived directly from `trend_direction`, which is in turn computed from `trend_pct`; both are excluded. The last-30-day and prior-30-day performance fields are also excluded because they construct or overlap the current-state decline proxy. `content_id` and `client_id` are excluded: they are identifiers, with `client_id` used only for grouping. No existing product decision flag or score is included.

The remaining fields are snapshot associations, so this notebook **does not** claim future prediction. For a future deployment model, each feature would need an explicit measurement cutoff strictly before a later label window. Missing values are median-imputed with indicators inside each training pipeline, avoiding test-set preprocessing leakage.

As a harness check, I deliberately add the forbidden `trend_pct` field under the same grouped split. If its metric is dramatically higher, that is evidence that the audit can detect the known label-derived shortcut—not a result to retain.

In [3]:
leakage_register = pd.DataFrame([
    ('trend_direction', 'label-derived', 'This field directly defines decline_proxy_rule.', 'excluded'),
    ('trend_pct', 'label-derived sibling', 'trend_direction is computed from this trend measure.', 'excluded'),
    ('impressions_last_30d / impressions_prev_30d', 'overlapping label inputs', 'These windows construct the decline comparison.', 'excluded'),
    ('clicks_last_30d / clicks_prev_30d', 'near-label window', 'Current/recent outcome window; not used for this proxy model.', 'excluded'),
    ('sessions_last_30d / sessions_prev_30d', 'near-label window', 'Current/recent outcome window; not used for this proxy model.', 'excluded'),
    ('content_id / client_id', 'identifier', 'IDs can enable memorization; client_id is grouping-only.', 'excluded from features'),
    ('product flags or decision scores', 'decision-derived', 'No such field is used as a model input.', 'not included'),
], columns=['field(s)', 'risk type', 'audit finding', 'treatment'])
display(leakage_register)

leaky_metrics, _, _ = evaluate_split('Grouped split with forbidden trend_pct', group_train, group_test, feature_cols + ['trend_pct'])
audit_comparison = pd.DataFrame([group_metrics, leaky_metrics])[['split', 'test_base_rate', 'roc_auc', 'average_precision', 'precision_at_20']]
for column in ['test_base_rate', 'roc_auc', 'average_precision', 'precision_at_20']:
    audit_comparison[column] = audit_comparison[column].map(lambda value: f'{value:.3f}')
display(audit_comparison)
print('The forbidden run is an audit demonstration only. trend_pct is removed from the retained model and from every reported performance claim.')

,field(s),risk type,audit finding,treatment
0,trend_direction,label-derived,This field directly defines decline_proxy_rule.,excluded
1,trend_pct,label-derived sibling,trend_direction is computed from this trend me...,excluded
2,impressions_last_30d / impressions_prev_30d,overlapping label inputs,These windows construct the decline comparison.,excluded
3,clicks_last_30d / clicks_prev_30d,near-label window,Current/recent outcome window; not used for th...,excluded
4,sessions_last_30d / sessions_prev_30d,near-label window,Current/recent outcome window; not used for th...,excluded
5,content_id / client_id,identifier,IDs can enable memorization; client_id is grou...,excluded from features
6,product flags or decision scores,decision-derived,No such field is used as a model input.,not included


,split,test_base_rate,roc_auc,average_precision,precision_at_20
0,After: client-grouped split,0.517,0.589,0.600,0.850
1,Grouped split with forbidden trend_pct,0.517,0.999,0.999,1.000


The forbidden run is an audit demonstration only. trend_pct is removed from the retained model and from every reported performance claim.


## 4. Error examples and claim rewrite

The examples below come only from held-out, pseudonymized clients. They omit identifiers, names, URLs, and raw queries. A false positive means the model assigned a high current-state proxy-decline probability but the proxy was not down; a false negative means the reverse. These examples are diagnostic, not explanations of why a page changed.

In [4]:
held_out = df.iloc[group_test].copy()
held_out['model_probability'] = grouped_probability
held_out['model_prediction'] = (held_out['model_probability'] >= 0.5).astype(int)
safe_columns = [target_col, 'model_probability', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'days_since_last_update', 'engagement_rate', 'content_type']
false_positives = held_out[(held_out.model_prediction == 1) & (held_out[target_col] == 0)].nlargest(3, 'model_probability')[safe_columns].copy().reset_index(drop=True)
false_negatives = held_out[(held_out.model_prediction == 0) & (held_out[target_col] == 1)].nsmallest(3, 'model_probability')[safe_columns].copy().reset_index(drop=True)
for error_frame in [false_positives, false_negatives]:
    error_frame['model_probability'] = error_frame['model_probability'].round(3)

print('False positives — high score, but the current-state proxy was not down:')
display(false_positives)
print('False negatives — low score, but the current-state proxy was down:')
display(false_negatives)
print('Observed pattern: the displayed false positives include pages with non-zero visibility but no proxy decline, while the displayed false negatives have very low 90-day impressions. These are descriptive examples only; they do not establish why an individual page changed.')
print('Displayed examples are the three most confident errors of each type; they show that observed snapshot signals do not perfectly reproduce the proxy label.')

False positives — high score, but the current-state proxy was not down:


,decline_proxy_rule,model_probability,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,engagement_rate,content_type
0,0,0.926,235,2,0.85,31.0,20,0.0,keyword article
1,0,0.894,2164,5,0.23,8.1,20,0.0,keyword article
2,0,0.889,1266,0,0.00,4.6,106,0.0,keyword article


False negatives — low score, but the current-state proxy was down:


,decline_proxy_rule,model_probability,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,engagement_rate,content_type
0,1,0.065,6,0,0.0,12.2,20,0.0,keyword article
1,1,0.067,13,0,0.0,11.5,20,0.0,keyword article
2,1,0.071,13,0,0.0,33.2,20,0.0,keyword article


Observed pattern: the displayed false positives include pages with non-zero visibility but no proxy decline, while the displayed false negatives have very low 90-day impressions. These are descriptive examples only; they do not establish why an individual page changed.
Displayed examples are the three most confident errors of each type; they show that observed snapshot signals do not perfectly reproduce the proxy label.


### Claim rewrite

**Earlier, too-strong wording:** “The model predicts declining content and should be used to decide what to refresh.”

**Public-safe rewrite:** “On this starter snapshot, the Logistic Regression model measured ranking signal for a current-state decline proxy on held-out pseudonymized clients. The result is directional and may support a human review queue; it does not predict a later outcome, establish why a page declines, or show that refreshing a page will improve performance.”

## Self-check

- [x] I named two paper findings and wrote constructive, concrete methodology questions.
- [x] I showed a before/after random-versus-client-grouped evaluation and test base rates.
- [x] I audited label-derived, window-overlap, identifier, and decision-derived leakage risks.
- [x] I ran a deliberate forbidden-feature check and did not retain it as a result.
- [x] I displayed public-safe held-out error examples with no IDs, names, URLs, or queries.
- [x] I rewrote the main claim using observed, measured, directional, and decision-support language.
- [x] I executed this notebook top to bottom and verified its outputs.
- [x] I committed this notebook to the repository.